# Homework14

Pre-Trained Transformer Models and Embeddings

## Goals

- Practice setting up classification and clustering modeling task from scratch
- Experiment with pre-trained transformer models for embedding and feature extraction
- Get more familiar with the `argmax()` and `pairwise_distance()` functions

### Setup

Run the following 2 cells to import all necessary libraries and helpers for this homework

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/Homework14/raw/refs/heads/main/Homework04_utils.py
!wget -qO- https://github.com/PSAM-5020-2026S-A/5020-utils/releases/latest/download/forest-tree.tar.gz | tar xz
!wget -qO- https://github.com/PSAM-5020-2026S-A/5020-utils/releases/latest/download/flowers102.tar.gz | tar xz

In [ ]:
import pandas as pd

from numpy import argsort
from os import listdir
from PIL import Image as PImage

from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import euclidean_distances

from torch import nn, Tensor, no_grad
from torch import float32 as t_float32, uint8 as t_uint8

from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms import v2

from transformers import CLIPModel, CLIPProcessor

from Homework04_utils import Homework04Utils

# Pre-Trained Models for Feature Extraction

We're going to repeat the very first image classification exercise we did in [`HW04`](https://github.com/PSAM-5020-2026S-A/Homework04), but this time we'll use a pre-trained model (and `PCA`) to help us extract features from our images.

The overall flow for doing this will be:
- Open images
- Get embeddings
- Get main principal component
- Plot and create a model
- Evaluate

In [ ]:
# Location of train/test files
TRAIN_PATH = "data/image/forest-tree/train"
TEST_PATH = "data/image/forest-tree/test"

# List comprehension for getting all of the filenames that end in "jpg" inside the train directory
train_files = sorted([f for f in listdir(TRAIN_PATH) if f.endswith("jpg")])
test_files = sorted([f for f in listdir(TEST_PATH) if f.endswith("jpg")])

In [ ]:
# Initialize an image feature extraction model
model_name = "openai/clip-vit-large-patch14"
preprocessor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to("cuda")

## Extract Features

So many features !!

Here we are going to leverage the full power of multi-dimensional tensors and parallelism to preprocess and run feature extraction on all $30$ train images and then all $150$ test images ***AT ONCE***.

If we just do something like:
`preprocessor(image_list)`

we'll get a list of tensors that we can pass to the model directly like this:
`model(tensor_list)`

and get a list of embeddings.

The result of running these commands on the training dataset will be a list with $30$ embeddings, where each embedding itself is a list of $768$ values.

In [ ]:
# Get list of images
train_imgs = [PImage.open(f"{TRAIN_PATH}/{fname}") for fname in train_files]

# Get list of tensors
train_t = preprocessor(images=train_imgs, return_tensors="pt").to("cuda")

# Get list of embeddings
with no_grad():
  train_embs = model.get_image_features(**train_t).pooler_output.cpu()

display(train_embs.shape)

In [ ]:
# TODO: repeat for test images

## One Feature To Rule Them All

We now have $768$ features for each of our images, but since we're building a model manually with some `if/else` statements like in `HW04`, we need to combine all of our features into a few numbers we can plot and extract information from.

`PCA`, FTW.

Let's create a `PCA` model to extract a single feature from each image.

In [ ]:
# TODO: create PCA
#       fit with the training data
#       transform train/test embeddings

## Visualize / Explore

This next cell is very similar to what we did in `HW04`.

It's the cell that separates a single value from each image and passes off the result to a function that will plot feature versus class label.

It assumes that the train and test principal components are in `train_pcs` and `test_pcs`, respectively.

In [ ]:
# list to keep info about image files
train_info = []

# iterate through pcs and filenames
for pcs,fname in zip(train_pcs, train_files):
  # TODO: get value of interest
  val = 0

  # store the info using the format specified above
  train_info.append([ val, fname ])

# plot the (val, filename) pairs
Homework04Utils.plot_labels_vals(train_info, "Train file feature")

## Create Model

Use the extracted value to create a classifier.

The class labels it has to return are: `"florist", "forest", "tree"`.

In [ ]:
# TODO: return a label based on embeddings and/or pcs
def awesome_classifier(pcs):
  # TODO: fill out this function !!!
  return "tree"

### Run on `train` and `test` data and check accuracy

This cell is already filled out, we just have to run it to get the accuracy values for our model.

In [ ]:
# lists to keep info about predictions
train_predictions = []
test_predictions = []

# iterate through train data
for pcs,fname in zip(train_pcs, train_files):
  prediction = clip_classifier(pcs)
  train_predictions.append([ prediction, fname ])

print("Train Accuracy")
print("  ", Homework04Utils.classification_accuracy(train_predictions))


# iterate through test data
for pcs,fname in zip(test_pcs, test_files):
  prediction = clip_classifier(pcs)
  test_predictions.append([ prediction, fname ])

print("\nTest Accuracy")
print("  ", Homework04Utils.classification_accuracy(test_predictions))

# One-Shot Classification

## Intro

We're going to leverage the general knowledge and patterns learned by pre-trained deep learning models to create a one-shot classifier model.

One-shot classifiers are models that learn how to describe/detect objects by just looking at one example from each class.

The overall flow for doing this using image embeddings can be something like:
- extract embeddings for all images in our dataset
- training: for each class, average a small number of embeddings to create class embeddings
- these embeddings now represent information about the classes we're trying to identify
- predicting: find the closest class embedding to each image embedding in the dataset

This is similar to one of the examples in our [Week 14](https://github.com/PSAM-5020-2026S-A/WK14) notebook where we used words to classify images.

## The Data

We're going to classify the [Oxford Flowers Dataset](https://www.robots.ox.ac.uk/~vgg/data/flowers/). This is a dataset made up of images of flowers and their names.

All of the images we're going to be working with should have been downloaded by the first cell, into separate `train` and `test` subdirectories inside `./data/image/flowers102/`.

The images all have the same (or similar) height, but very different widths. Depending on the architecture/model that we choose to use for  embedding, this might be something we have to standardize. Transformer architectures and their preprocessors will automatically deal with these differences, where CNN models are a bit more strict.

Let's start by defining a function that helps parse the classification label from file names or paths.

In [ ]:
def filepath_to_label(filepath):
  return filepath.split("/")[-1].split("_")[-1].split(".")[0]

## The Model

Now we have to define an embedding model, and (probably) a pre-processing strategy.

What wee need here is a pre-trained model that is able to turn images of various sizes into feature lists of fixed-length.

Our [Week 14](https://github.com/PSAM-5020-2026S-A/WK14) notebook has a couple of examples of how to use `ResNet`, `CLIP` and `SigLIP` models to do this, but there are other options that could be used. Since we're not doing any text processing, any kind of deep learning visual model can (theoretically) be used.

Some other examples: [Nomic Vision](https://huggingface.co/nomic-ai/nomic-embed-vision-v1.5), [EfficientNet](https://pytorch.org/hub/nvidia_deeplearningexamples_efficientnet/), [ViT](https://huggingface.co/google/vit-base-patch16-224-in21k), [DINOv3](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m), etc.

In [ ]:
# TODO: define model and pre-processing routine/function/strategy for images

## Train Data

Now we process the train data. Fun.

There are many ways to do this, but one possible strategy is to go through all of the files inside the `./data/image/flowers102/train` directory and append each image's label and embedding to separate lists, called `train_labels` and `train_embeddings`.

Then, create a `DataFrame` using the embeddings and add the labels to the same `DataFrame`.

Depending on the model chosen, this can take a few minutes.

In [ ]:
# TODO: extract labels and embeddings for each image in the training dataset

TRAIN_DIR = "./data/image/flowers102/train"

train_fnames = sorted([f for f in listdir(TRAIN_DIR) if f.endswith("jpg")])

train_labels = []
train_embeddings = []

for fname in train_fnames:
  # TODO: replace this with code
  pass

# TODO: combine the lists into a DataFrame

### Train Data Questions

<span style="color:hotpink;">
How many images are in the training dataset ?<br>
How many <em>"features"</em> ?<br>
How many unique classes do we have for this data ?
</span>

<span style="color:hotpink;">ADD ANSWER TO THIS CELL</span>

## Test Data

Repeat the above process for the files inside the `./data/image/flowers102/test` directory: append each image's label and embedding to separate lists, called `test_labels` and `test_embeddings`.

Since we won't do any other kind of processing on this data, it's not as important to combine the labels and embeddings into a `DataFrame`.

And again, this can take a few minutes.

In [ ]:
# TODO: Repeat label and embedding extraction for all images in the test dataset

### Test Data Questions

<span style="color:hotpink;">
How many images are in the test dataset ?<br>
Anything odd or unusual about this ?<br>
Is the test dataset balanced ?<br>
Does it matter ?
</span>

<span style="color:hotpink;">ADD ANSWER TO THIS CELL</span>

## Train the model

This is the unconventional part. We don't need to train any models, but use the already-trained one to derive some information about our training data that can then be used to make new predictions.

There are different ways to do this, but a recommended strategy here could be:
- get a list of unique labels in our dataset
- iterate over the labels, filter the `DataFrame` by label and compute an average embedding for all images of each label
- these are now class embeddings, as they hold aggregate information about multiple instances of each class
- we should end up with as many class embeddings as there are unique labels in the dataset

In [ ]:
# TODO: create class embeddings by averaging embeddings for all images of each label

## Predict and Evaluate

We have class embeddings, and we have instance embeddings from all images in the test dataset.

We can use the `euclidean_distances()` function to calculate pairwise distances between each image and each class embedding.

Then, we'll go through each image and get the index of the class embedding that is closest to its image embedding.

We can create a `predictions` list to compare to the `test_labels` list we extracted above.

In [ ]:
# TODO: use euclidean_distances() and argsort() to determine the closest class for each image

In [ ]:
print(classification_report(test_labels, test_preds))

### Interpretation

<span style="color:hotpink;">
So... What happened ?<br>
What are some advantages and disadvantages of using this strategy for classification ?
</span>

<span style="color:hotpink;">ADD ANSWER TO THIS CELL</span>